
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - From AI Playground to Deployment
In this demo, you will learn how to move from rapid prototyping in the AI Playground to a fully deployed agent in Databricks. Starting with an exported Playground notebook, you’ll build an agent that can call Unity Catalog functions and vector search endpoints, test its responses, evaluate its performance with MLflow's GenAI Evaluation tools, and deploy it for production use. This process demonstrates the full lifecycle of an AI agent—from experimentation to deployment—showing how Databricks unifies tool creation, testing, evaluation, and deployment into a single streamlined workflow.

### Learning Objectives
_By the end of this demo, you will be able to:_ 
- Inspect Unity Catalog and workspace assets that power agent tooling
- Understand and modify a Playground-exported agent notebook
- Build a tool-calling agent with MLflow's `ResponsesAgent`
- Manually test and evaluate the agent using MLflow Trace UI - and Mosaic AI Evaluation
- Log and register the agent to Unity Catalog for governance and reproducibility

## Demo Scenario

You are building an intelligent customer service agent for an e-commerce company. The agent needs to handle customer inquiries about returns, refunds, and product support. To make the agent effective, you'll create specialized tools that can:

- Access customer service data to retrieve recent return requests
- Look up company policies to ensure compliance with business rules
- Review customer order history to determine eligibility for returns
- Search product documentation to provide technical support

This scenario demonstrates how to combine structured data queries with unstructured document search to create a comprehensive customer service solution using Unity Catalog functions as agent tools.

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## REQUIRED - Classroom Setup

Run the following cell to configure your working environment for this notebook.

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course in your environment.

In [0]:
%run "../../Includes/Classroom-Setup-3.1"

## A. Inspecting UC & Workspace Assets

#### Unity Catalog Assets
As a part of the setup for this demonstration, multiple functions have been created. Click on the **Catalog** on the left menu and navigate to the your user catalog > default. In the default schema you will find four functions that have been created. Here is a summary of those SQL functions, but you can also view them in the UI by opening **Catalog** in a new window and navigating to `<labuserXXXXXXXX_XXXXXXXXXX>.default` to inspect the metadata of the functions. 

1. `get_latest_return`: Returns the most recent customer service interaction, such as returns.
1. `get_order_history`: This takes the user_name of a customer as an input and returns the number of returns and the issue category
1. `get_return_policy`: Returns the details of the Return Policy
1. `search_product_docs`: Searches product documentation using vector search to retrieve relevant documentation excerpts for troubleshooting and support. This should be used to search by product as each product has its own documentation.

#### Workspace Assets
- As a part of this classroom setup, a **standard vector search endpoint** has been provisioned. You have access to query this endpoint, but you do not have the ability to see the indexed table due to limitations with the lab environment.

> This course does not go into detail on deploying a vector search endpoint. Please see our [other](https://www.databricks.com/training/catalog?search=generative+ai+solution+development) course offerings on this topic.

- As part of this classroom setup, an agent has been pre-deployed to a model serving endpoint. We will recreate the agent later in the course, but due to time constraints, we will be querying this pre-deployed agent for this lab.

> This course does not go into detail on deploying an (agent) model to Mosaic AI Model Serving. Please see our [official documentation](https://docs.databricks.com/aws/en/machine-learning/model-serving/) and course [Generative AI Application Deployment and Monitoring](https://www.databricks.com/training/catalog/generative-ai-application-deployment-and-monitoring-2713) for more information.

## B. Inspecting and Understanding the Contents of an Agent Notebook from Playground

After enabling an agent to use tools from Unity Catalog (see [Section D]($../M02 - Building AI Agents on Databricks/2.2 - Build Agent Tools and Prototype with AI Playground); full instructions are also provided in the Appendix), you can export a notebook to begin customizing and tuning your agent. Before doing so, let’s walk through a sample notebook that illustrates a typical starting point for development.

As a part of the lab setup, there is a **[notebook]($./Agent Files/driver)** that has been exported from the Playground. This notebook builds a tool-calling agent on Databricks using MLflow's `ResponsesAgent`. It shows the full lifecycle:
1. Author an agent that can call tools (Unity Catalog functions, optional Vector Search retrievers) via an OpenAI-compatible endpoint hosted by Databricks.
1. Manually test the agent (predict + streaming).
1. Evaluate the agent with MLflow GenAI Evaluation scorers.
1. Log the agent to MLflow with required workspace resources.
1. Register to Unity Catalog.
1. Deploy the agent with `databricks.agents.` (This part is skipped due to time constraints for this course). 

### Instructions
1. Navigate to the folder called **Agent Files** using the folder icon in the left sidebar menu. It is located in the same folder as  where this notebook is located (or you can click [here]($./Agent Files/driver))
1. In there you will find two files:
    - `driver`: This is a notebook that looks very similar to the notebook that comes from Playground. It has been especially configured for this lab environment. 
    - `env.json`: This file is blank unless (unless you have already ran the `driver` notebook) but it will eventually contain your person labuser catalog after running `driver`. 
1. Open the `driver` notebook and click the **Run all** button at the top right of the notebook. 
    - The contents of this notebook walk through how to use the tools you built with your agent from the Playground. 
        - This is an auto-generated notebook created by an AI playground export. In this notebook, you will:
            - Author a tool-calling [MLflow's `ResponsesAgent`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.pyfunc.html#mlflow.pyfunc.ResponsesAgent) that uses the OpenAI client
            - Manually test the agent's output
            - Evaluate the agent with [Mosaic AI Agent Evaluation](https://www.databricks.com/product/machine-learning/retrieval-augmented-generation)
            - Log and deploy the agent

> Note: once you run this notebook, you will find an additional `.py` file called `agent.py`. This is created via the `driver` notebook. 

## C. Agent Evaluation, MLflow, and Your UC-Registered Agent
After successfully running the notebook `driver`, let's view the evaluation output. Use the table of contents icon on the left side menu and navigate to **Evaluate the agent with Agent Evaluation**.
> This course does _not_ perform a deep dive on MLflow. Please see the course [Generative AI Application Evaluation and Governance](https://www.databricks.com/training/catalog/generative-ai-application-evaluation-and-governance-2668) for further training on this topic. 
### Instructions
Let's investigate the output for **cell 15**.  

1. As a part of the output, there will be a blue button that says **View evaluation results**. Click on it. This will open a new window within Databricks that shows the evaluation results within MLflow. Explore the outputs there and navigate back to the `driver` notebook.

2. The output for **cell 15** also shows the trace breakdown, input/output prompts, attributes, events, and assessments as a part of the evaluation process. Notice that the assessment shows the rationale behind the Feedback assessment . The screenshot below shows a typical output. 
<img src="../../Includes/images/mosaic-ai-framework.png" width="1000"/>
3. Click on one of the completions and take a look at some of the thoughts being used in the chat tab. For example, in the screenshot below we see that in the first completion, the assistant chose to search for relevant product documentation.
<img src="../../Includes/images/mosaic-ai-framework2.png" width="500"/>
4. This notebook also performs a pre-deployment validation step with sample input data in cell 17, which is best practice before registering to Unity Catalog. 
5. Your (agent) model has been registered to Unity Catalog in location `<labuserXXXXXXXX_XXXXXXXXXX>.default.my_first_agent`. If you navigate there using the **Catalog** menu, you will now see you have a model as a part of running **cell 19**.

### C1. Querying an Agent Deployed to Mosaic AI Model Serving. 
Our notebook `driver` stops short of deploying the model. We have a model already available that we can query, which we will do next.
#### Instructions
1. Navigate to **Serving** on the left side menu and click on the model called `agents_dbacademy-datasets-demo-agent-model`.
1. Click on **Use** at the top right. 
1. Begin querying! Here are some examples of queries you can use: 
    - _Get the order history of @ronald54@example.net_
    - _Can you tell me the policy for exchanging items?_

## D. Exporting Your Own Agent Notebook
Once you have enabled an agent to use tools from UC (see the appendix for a refresher) you can export a notebook and begin developing in your own environment using the notebook as a template.  
 
### Instructions
1. After selecting all the tools from `<labuserXXXXXXXX_XXXXXXXXXX>.default` for Claude Sonnet 3.7 in the AI Playground, navigate to **Get code** at the top of the AI Playground. 
1. Select **Create Agent Notebook**.
1. This will open a new window showing you pre-generated Python code. 
    - Selecting **Create Agent Notebook** creates a new directory in your user notebook with path **Workspace/Users/labuserXXXXXXXX_XXXXXXXXXX/new_agent_code_folder**.
1. Inspect the notebook and read the description of what the notebook accomplishes. It is very similar to our `driver` notebook mentioned above, but our notebook has been specially configured for this lab environment. 


## Conclusion

You have now completed the journey from Playground experimentation to agent deployment. Along the way, you explored Unity Catalog functions, tested agent behavior, and applied evaluation tools to measure quality. Finally, you registered and deployed your agent, making it accessible and reusable within Databricks. These steps illustrate how Databricks supports the entire agent development lifecycle by bridging experimentation, evaluation, and deployment.

# Appendix 
For completeness, we record how to enable Tooling in the AI Playground. 

## D. Enabling Tooling in AI Playground
### 1. Open the AI Playground

1. In the Databricks workspace, navigate to **Playground** via the left-hand navigation pane under **AI/ML**.
2. Select an LLM model—be sure it has the **“Tools enabled”** label to allow tool activation. For example, select **Claude Sonnet 3.7**. Here is an image for reference: 

<img src="../../Includes/images/claude-tool.png" width="600"/>


_Note the tool icon next to the model selection._

### 2. Add Tools to Your Agent
After selecting your agent, you can now add a tool in the Playground. Here is an image for reference: 

<img src="../../Includes/images/tool-select.png" alt="Tool Selection" width="600"/>


1. In the **Tools** menu, select **Add**.
1. Select **+ Add tool**.
1. Under the  **UC Function** tab, click the dropdown menu labeled **Add hosted function**.
    - Select `<labuserXXXXXXXX_XXXXXXXXXX>.default.search_product_docs`
    - Click on **Save**.
    - Repeat for the other functions.
    - **`Warning:`** Do not use  `<labuserXXXXXXXX_XXXXXXXXXX>.default.*` for this setup.

1. **Test the Functions**
   - In the chat window, type a prompt that would require the agent to use one or more of your tools.  
     Example prompts:
     - “Show me the latest customer return.”
     - “I want to return the product I purchased recently.”
   - The agent doesn't know your account, so it will ask for your email. Provide the email for `nicolas.pelaez@example.com` as an example. When asked about the product, use `bluetooth headphone` and it should return the right order.
   - Follow the conversation and see if you are eligible for return or not.

5. **Review the Output**
   - Check the agent’s response and verify that the output matches the expected results from your functions.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>